# LLM-as-Judge: Entity Code Behavioral Verification

Two-pass Gemini judge that verifies each generated entity's code against:
1. Its causal role in the dependency graph
2. Required metric attributes from metric contracts
3. *(Pass 2)* Cross-entity interface compatibility for flagged pairs

**Kernel**: Use the backend env — `engine/backend/env/bin/python`

## ⚙️ Config — edit this cell

In [ ]:
import os

# Path to the exported code-workspace directory
WORKSPACE_PATH = "code-workspace-code_gen-749f346ee2ca4dadb6539e987a59dcbf-2026-05-04_05-14-11"

# Optional: path to a standalone causal JSON file.
# Supports three formats:
#   - Exported causal bundle  (export_type == "causal_bundle")
#   - Flat list               ([{head, relationship, tail, detail}])
#   - inputs.json             (causalData string key)
# If None, loads from workspace/checkpoints/inputs.json automatically.
CAUSAL_FILE_PATH = "/Users/rapepong/Downloads/response_raw_gemini_combined-bundle.json.json-bundle-2026-05-04T13-38-01-706Z.json"

# --- Auth: pick ONE method ---
# Method A: API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# Method B: Application Default Credentials (ADC)
# Run once in terminal:  gcloud auth application-default login
# Then set USE_ADC = True. Routes through Vertex AI — set project + location.
USE_ADC = True
GCP_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT", "senior-cpe-04")   # e.g. "my-gcp-project"
GCP_LOCATION = os.getenv("VERTEX_AI_LOCATION", "us-central1")

MODEL = "gemini-2.5-pro"  # or gemini-2.5-pro for harder verdicts

# Pass 2 threshold: entities with behavior_score <= this go to interface audit
PASS2_SCORE_THRESHOLD = 3

# Output directory for verdict JSONs
OUTPUT_DIR = "judge_output"

## 📦 Imports

In [2]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from google import genai
from google.genai.types import GenerateContentConfig
from IPython.display import display

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("google-genai version:", genai.__version__)

google-genai version: 1.69.0


## 📂 Load Workspace Data

In [3]:
ws = Path(WORKSPACE_PATH)
artifacts = ws / "artifacts"
checkpoints = ws / "checkpoints"

# --- Entity dependency graph ---
deps_path = checkpoints / "state1c_entity_dependencies.json"
with open(deps_path) as f:
    deps_data = json.load(f)
dep_edges = deps_data["edges"]   # [{from, to, reason}]
entity_order = deps_data.get("order", [])
print(f"Dependency edges: {len(dep_edges)}")

# --- Metric contracts ---
with open(artifacts / "metric_contracts.json") as f:
    mc_data = json.load(f)
metrics = mc_data["metrics"]  # [{name, entities, required_attrs, rationale}]
print(f"Metric contracts: {len(metrics)}")

# --- Causal triples ---
def _extract_triples_from_chunks(chunks: list) -> list[dict]:
    """Walk chunk → classes → extracted and collect all {head, relationship, tail, detail} triples."""
    triples = []
    for chunk in chunks:
        for cls in chunk.get("classes", []):
            for e in cls.get("extracted", []):
                triples.append(e)
    return triples

def load_causal_triples(causal_file_path=None) -> list[dict]:
    """
    Load causal triples from:
      1. Exported bundle JSON  (export_type == "causal_bundle", key: raw_extraction)
      2. Flat list             ([{head, relationship, tail, detail}, ...])
      3. inputs.json           (causalData string containing JSON chunks)
    Falls back to workspace/checkpoints/inputs.json when causal_file_path is None.
    """
    if causal_file_path:
        with open(causal_file_path) as f:
            raw = json.load(f)

        # Format 1: exported causal bundle
        if isinstance(raw, dict) and raw.get("export_type") == "causal_bundle":
            return _extract_triples_from_chunks(raw.get("raw_extraction", []))

        # Format 2: flat list of triples
        if isinstance(raw, list) and raw and "head" in raw[0]:
            return raw

        # Format 3: inputs.json dict (causalData string)
        if isinstance(raw, dict) and "causalData" in raw:
            causal_str = raw["causalData"]
        else:
            raise ValueError(f"Unrecognised causal file format. Top-level keys: {list(raw.keys()) if isinstance(raw, dict) else type(raw)}")
    else:
        inputs_path = checkpoints / "inputs.json"
        with open(inputs_path) as f:
            inp = json.load(f)
        causal_str = inp.get("causalData", "")

    # Parse causalData string (strip optional markdown comment header)
    if isinstance(causal_str, str):
        lines = causal_str.strip().split("\n", 1)
        json_str = lines[1] if len(lines) > 1 and lines[0].startswith("#") else causal_str
        chunks = json.loads(json_str)
    else:
        chunks = causal_str  # already parsed

    return _extract_triples_from_chunks(chunks)

causal_triples = load_causal_triples(CAUSAL_FILE_PATH)
print(f"Causal triples: {len(causal_triples)}")

# --- Entity source files ---
entity_files = sorted((artifacts / "entities").glob("*.py"))
print(f"Entity files: {len(entity_files)}")

Dependency edges: 18
Metric contracts: 8
Causal triples: 93
Entity files: 16


## 🗂️ Build Entity Index

In [4]:
NOISE_PREFIXES = {"entity", "canonical", "manual", "the"}

def entity_name_from_file(path: Path, label_map: dict[str, str] | None = None) -> str:
    """
    Return clean human-readable name for an entity file.
    Priority: explicit label from metadata label_map > filename parsing.
    """
    eid = path.stem
    if label_map and eid in label_map:
        return label_map[eid].lower()

    parts = eid.split("-")
    name_parts = []
    for p in parts:
        if p in NOISE_PREFIXES:
            continue
        if p.isdigit():
            continue
        if len(p) > 10 and p.isalnum():  # uuid / timestamp segment
            continue
        name_parts.append(p)
    return " ".join(name_parts)

def fuzzy_match(contract_name: str, entity_name: str, threshold: int = 1) -> bool:
    """
    True if meaningful keyword overlap between contract_name and entity_name >= threshold.
    Uses a broad stopword set to avoid false matches on ubiquitous words like 'waste'.
    """
    stop = {
        "the", "a", "an", "of", "at", "in", "and", "or", "to", "for",
        "from", "by", "with", "is", "are", "was", "be", "as",
    }
    def sig(s: str) -> set[str]:
        return set(s.lower().replace("-", " ").split()) - stop

    contract_sig = sig(contract_name)
    entity_sig = sig(entity_name)
    return len(contract_sig & entity_sig) >= threshold

def triple_match(triple: dict, entity_name: str) -> bool:
    """
    Match a causal triple to an entity.
    Requires 2+ keyword overlap OR exact substring, to avoid 'waste'-floods.
    """
    head = triple.get("head", "").lower()
    tail = triple.get("tail", "").lower()
    en = entity_name.lower()
    en_words = set(en.split())

    for text in (head, tail):
        # Exact entity name substring
        if en in text or text in en:
            return True
        # 2+ keyword overlap (stricter than fuzzy_match for triples)
        text_words = set(text.replace("-", " ").split())
        stop = {"the", "a", "an", "of", "at", "in", "and", "or", "to", "for"}
        overlap = (en_words - stop) & (text_words - stop)
        if len(overlap) >= 2:
            return True
        # Single overlap only if entity name is short (bma, n15 etc)
        if len(en_words) == 1 and overlap:
            return True
    return False

# Load explicit label map from inputs.json (more reliable than filename parsing)
label_map: dict[str, str] = {}
try:
    with open(checkpoints / "inputs.json") as f:
        inp = json.load(f)
    for entry in inp.get("userEntityList", []):
        label_map[entry["id"]] = entry["label"]
    print(f"Loaded {len(label_map)} entity labels from inputs.json")
except Exception as e:
    print(f"Could not load label map from inputs.json: {e}")

# Build per-entity context index
entity_index: dict[str, dict] = {}

for ef in entity_files:
    eid = ef.stem
    human_name = entity_name_from_file(ef, label_map)

    outgoing = [e for e in dep_edges if e["from"] == eid]
    incoming = [e for e in dep_edges if e["to"] == eid]

    required_attrs: list[dict] = []
    for m in metrics:
        matched = [me for me in m["entities"] if fuzzy_match(me, human_name)]
        if matched:
            required_attrs.append({
                "metric": m["name"],
                "label": m["label"],
                "matched_as": matched,
                "attrs": m.get("required_attrs", []),
                "rationale": m.get("rationale", ""),
            })

    relevant_triples = [t for t in causal_triples if triple_match(t, human_name)]

    entity_index[eid] = {
        "eid": eid,
        "human_name": human_name,
        "file": ef,
        "outgoing": outgoing,
        "incoming": incoming,
        "required_attrs": required_attrs,
        "causal_triples": relevant_triples,
    }

print(f"\nIndexed {len(entity_index)} entities:")
for eid, ctx in entity_index.items():
    print(f"  [{ctx['human_name']!r:35s}] out={len(ctx['outgoing'])} in={len(ctx['incoming'])} metrics={len(ctx['required_attrs'])} triples={len(ctx['causal_triples'])}")

Loaded 16 entity labels from inputs.json

Indexed 16 entities:
  ['the market'                       ] out=0 in=1 metrics=0 triples=1
  ['the school'                       ] out=0 in=1 metrics=0 triples=1
  ['the waste cage'                   ] out=2 in=1 metrics=2 triples=9
  ['mirror foundation'                ] out=0 in=1 metrics=0 triples=2
  ['waste bins'                       ] out=0 in=1 metrics=2 triples=8
  ['waste collectors'                 ] out=1 in=0 metrics=2 triples=6
  ['staff'                            ] out=2 in=1 metrics=1 triples=6
  ['housekeepers'                     ] out=5 in=0 metrics=1 triples=8
  ['students'                         ] out=2 in=1 metrics=1 triples=4
  ['bma'                              ] out=0 in=2 metrics=0 triples=2
  ['faculties'                        ] out=1 in=0 metrics=0 triples=2
  ['n15'                              ] out=0 in=1 metrics=0 triples=1
  ['waste'                            ] out=4 in=6 metrics=2 triples=57
  ['sorting f

## 📝 Prompt Generator

In [5]:
PASS1_SYSTEM = """\
You are an expert simulation engineer auditing auto-generated Python entity classes.
Each entity is a component in a discrete-event waste management simulation.
Your job: verify that the entity's code faithfully implements its causal role.

## Scoring rubric for behavior_score (0-3)
- 3: All causal roles implemented with correct state transitions and data flow
- 2: Roles present but incomplete (e.g., method exists but logic is shallow)
- 1: Partially missing (some roles implemented, others absent)
- 0: Causal role largely absent from code

## Output format
Return ONLY valid JSON matching the schema provided.
"""

PASS1_SCHEMA = {
    "type": "object",
    "properties": {
        "entity": {"type": "string"},
        "behavior_score": {"type": "integer"},
        "verdict": {"type": "string", "enum": ["pass", "warn", "fail"]},
        "causal_claim_checks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "claim": {"type": "string"},
                    "status": {"type": "string", "enum": ["FOUND", "PARTIAL", "MISSING"]},
                    "evidence": {"type": "string"},
                },
                "required": ["claim", "status", "evidence"],
            },
        },
        "missing_attrs": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Metric attrs required by contracts but absent or wrong in on_query()",
        },
        "interface_concerns": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Potential cross-entity interface mismatches spotted in this entity",
        },
        "summary": {"type": "string"},
    },
    "required": ["entity", "behavior_score", "verdict", "causal_claim_checks", "missing_attrs", "interface_concerns", "summary"],
}

def build_causal_claims(ctx: dict) -> str:
    """Build numbered causal claim list for the judge to verify."""
    lines = []
    idx = 1
    for e in ctx["outgoing"]:
        to_name = e["to"].replace("-", " ")
        lines.append(f"{idx}. (OUTGOING) \"{ctx['human_name']} → {to_name}\": {e['reason']}")
        idx += 1
    for e in ctx["incoming"]:
        from_name = e["from"].replace("-", " ")
        lines.append(f"{idx}. (INCOMING) \"{from_name} → {ctx['human_name']}\": {e['reason']}")
        idx += 1
    for t in ctx["causal_triples"][:10]:  # cap to avoid prompt bloat
        lines.append(f"{idx}. (CAUSAL TRIPLE) \"{t['head']}\" --[{t['relationship']}]--> \"{t['tail']}\" {t.get('detail', '')}")
        idx += 1
    return "\n".join(lines) if lines else "No causal claims found for this entity."

def build_metric_section(ctx: dict) -> str:
    if not ctx["required_attrs"]:
        return "No metric contracts reference this entity."
    lines = []
    for m in ctx["required_attrs"]:
        attr_list = ", ".join(a["attr"] for a in m["attrs"]) if m["attrs"] else "(none specified)"
        lines.append(f"- Metric `{m['metric']}` ({m['label']}): required attrs = [{attr_list}]")
        lines.append(f"  Rationale: {m['rationale']}")
    return "\n".join(lines)

def build_pass1_prompt(ctx: dict) -> str:
    code = ctx["file"].read_text()
    causal_claims = build_causal_claims(ctx)
    metric_section = build_metric_section(ctx)

    return f"""{PASS1_SYSTEM}

## Entity: `{ctx['eid']}` (human name: {ctx['human_name']})

---
### Causal Claims to Verify
For each claim below, find evidence in the code and rate: FOUND | PARTIAL | MISSING.
One sentence of evidence per claim.

{causal_claims}

---
### Metric Attribute Contracts
The `on_query()` method must expose these attributes for metrics to work.
Check that each attr name exists and holds meaningful data.

{metric_section}

---
### Entity Source Code

```python
{code}
```

---
Audit the code against the causal claims and metric contracts above.
Return JSON only.
"""

# Preview one prompt
sample_eid = list(entity_index.keys())[0]
sample_prompt = build_pass1_prompt(entity_index[sample_eid])
print(f"Sample prompt for {sample_eid} — {len(sample_prompt)} chars")
print(sample_prompt[:1200], "\n...(truncated)")

Sample prompt for entity-117-the-market — 4005 chars
You are an expert simulation engineer auditing auto-generated Python entity classes.
Each entity is a component in a discrete-event waste management simulation.
Your job: verify that the entity's code faithfully implements its causal role.

## Scoring rubric for behavior_score (0-3)
- 3: All causal roles implemented with correct state transitions and data flow
- 2: Roles present but incomplete (e.g., method exists but logic is shallow)
- 1: Partially missing (some roles implemented, others absent)
- 0: Causal role largely absent from code

## Output format
Return ONLY valid JSON matching the schema provided.


## Entity: `entity-117-the-market` (human name: the market)

---
### Causal Claims to Verify
For each claim below, find evidence in the code and rate: FOUND | PARTIAL | MISSING.
One sentence of evidence per claim.

1. (INCOMING) "entity 123 the waste cage → the market": the market contributes to waste overflow at the waste cage

## 🤖 Gemini Client Setup

In [6]:
if USE_ADC:
    if not GCP_PROJECT:
        raise ValueError("Set GCP_PROJECT when using ADC (Vertex AI requires a project)")
    client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)
    print(f"Auth: ADC (Vertex AI) — project={GCP_PROJECT} location={GCP_LOCATION}")
else:
    if not GEMINI_API_KEY:
        raise ValueError("Set GEMINI_API_KEY or enable USE_ADC in the Config cell")
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Auth: API key")

def call_gemini(prompt: str, schema: dict) -> dict:
    """Single Gemini call with JSON schema output."""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=schema,
            temperature=0.1,
        ),
    )
    return json.loads(response.text)

print(f"Gemini client ready — model={MODEL}")

Auth: ADC (Vertex AI) — project=senior-cpe-04 location=us-central1
Gemini client ready — model=gemini-2.5-pro


## 🔍 Pass 1 — Per-Entity Contract Audit

In [ ]:
import time

pass1_results: dict[str, dict] = {}
pass1_errors: dict[str, str] = {}

eids = list(entity_index.keys())
print(f"Running Pass 1 on {len(eids)} entities...\n")

for i, eid in enumerate(eids, 1):
    ctx = entity_index[eid]
    print(f"[{i}/{len(eids)}] {eid} ... ", end="", flush=True)
    try:
        prompt = build_pass1_prompt(ctx)
        result = call_gemini(prompt, PASS1_SCHEMA)
        pass1_results[eid] = result

        # Save individual verdict
        out_path = Path(OUTPUT_DIR) / f"pass1_{eid}.json"
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)

        score = result.get("behavior_score", "?")
        verdict = result.get("verdict", "?")
        print(f"{verdict.upper()} (score={score})")
    except Exception as exc:
        pass1_errors[eid] = str(exc)
        print(f"ERROR: {exc}")

    # Rate limit buffer
    if i < len(eids):
        time.sleep(1.5)

print(f"\nDone. {len(pass1_results)} succeeded, {len(pass1_errors)} errors.")

Running Pass 1 on 16 entities...

[1/16] entity-117-the-market ... FAIL (score=1)
[2/16] entity-118-the-school ... FAIL (score=1)
[3/16] entity-123-the-waste-cage ... WARN (score=2)
[4/16] entity-14-mirror-foundation ... PASS (score=3)
[5/16] entity-143-waste-bins ... WARN (score=2)
[6/16] entity-145-waste-collectors ... FAIL (score=2)
[7/16] entity-19-staff ... FAIL (score=1)
[8/16] entity-2-housekeepers ... WARN (score=2)
[9/16] entity-20-students ... WARN (score=1)
[10/16] entity-27-bma ... PASS (score=3)
[11/16] entity-53-faculties ... WARN (score=2)
[12/16] entity-81-n15 ... PASS (score=3)
[13/16] entity-canonical-0-waste ... FAIL (score=1)
[14/16] entity-manual-1777470690931-sorting-facility ... WARN (score=3)
[15/16] entity-manual-1777470711550-garbage-truck ... PASS (score=3)
[16/16] entity-manual-1777478778843-event ... FAIL (score=1)

Done. 16 succeeded, 0 errors.


import json, pandas as pd
from pathlib import Path

# ── Reload source: pick ONE ──────────────────────────────────────────────────
# Option A: CSV export (e.g. from a previous run saved manually)
PASS1_CSV = "/Users/rapepong/Development/kmutt/Garbage-Management-Simulation-Engine/Experiment/code_generation/entity_design/enitiy_code_verification/llmjudgepass1_best.csv"

# Option B: individual JSON verdicts in judge_output/
PASS1_JSON_DIR = OUTPUT_DIR   # set to None to skip
# ─────────────────────────────────────────────────────────────────────────────

pass1_results: pd.DataFrame | dict = {}
pass1_errors: dict[str, str] = {}

if PASS1_CSV and Path(PASS1_CSV).exists():
    pass1_results = pd.read_csv(PASS1_CSV)
    print(f"Loaded {len(pass1_results)} rows from CSV: {PASS1_CSV}")
    print(pass1_results[["entity", "score", "verdict"]].to_string(index=False))

elif PASS1_JSON_DIR:
    _dict: dict[str, dict] = {}
    for f in sorted(Path(PASS1_JSON_DIR).glob("pass1_*.json")):
        eid = f.stem.removeprefix("pass1_")
        with open(f) as fh:
            _dict[eid] = json.load(fh)
    pass1_results = _dict
    print(f"Loaded {len(pass1_results)} results from JSON dir: {PASS1_JSON_DIR}")
    for eid, r in pass1_results.items():
        print(f"  {eid}: score={r.get('behavior_score','?')} verdict={r.get('verdict','?')}")

else:
    print("No reload source configured — run Pass 1 cells to populate pass1_results")

In [ ]:
import json, re
from pathlib import Path

# Reconstruct pass1_results dict from saved individual verdict JSONs.
# Run this instead of Pass 1 after a runtime restart.
pass1_results: dict[str, dict] = {}
pass1_errors: dict[str, str] = {}

for f in sorted(Path(OUTPUT_DIR).glob("pass1_*.json")):
    # filename: pass1_entity-2-housekeepers.json → eid: entity-2-housekeepers
    eid = f.stem.removeprefix("pass1_")
    with open(f) as fh:
        pass1_results[eid] = json.load(fh)

print(f"Reloaded {len(pass1_results)} Pass 1 results from {OUTPUT_DIR}/")
for eid, r in pass1_results.items():
    print(f"  {eid}: score={r.get('behavior_score','?')} verdict={r.get('verdict','?')}")

## 💾 Reload Pass 1 Results (run this after runtime restart instead of re-running Pass 1)

In [ ]:
def verdict_emoji(v: str) -> str:
    return {"pass": "✅", "warn": "⚠️", "fail": "❌"}.get(v, "❓")

rows = []
for eid, r in pass1_results.items():
    claims = r.get("causal_claim_checks", [])
    found = sum(1 for c in claims if c["status"] == "FOUND")
    partial = sum(1 for c in claims if c["status"] == "PARTIAL")
    missing = sum(1 for c in claims if c["status"] == "MISSING")
    rows.append({
        "entity": eid,
        "verdict": f"{verdict_emoji(r.get('verdict','?'))} {r.get('verdict','?')}",
        "score": r.get("behavior_score", "?"),
        "claims (F/P/M)": f"{found}/{partial}/{missing}",
        "missing_attrs": len(r.get("missing_attrs", [])),
        "interface_concerns": len(r.get("interface_concerns", [])),
        "summary": r.get("summary", "")[:120],
    })

df = pd.DataFrame(rows).sort_values("score")
pd.set_option("display.max_colwidth", 130)
display(df.reset_index(drop=True))

if pass1_errors:
    print("\nErrors:")
    for eid, err in pass1_errors.items():
        print(f"  {eid}: {err}")

,entity,verdict,score,claims (F/P/M),missing_attrs,interface_concerns,summary
0,entity-117-the-market,❌ fail,1,1/0/1,0,0,"The entity correctly implements waste generation at the 'Red Building area' as per one causal claim. However, it complet"
1,entity-118-the-school,❌ fail,1,0/1/1,0,1,"The entity correctly models continuous waste generation. However, it fails to implement the specified causal link to the"
2,entity-19-staff,❌ fail,1,2/2/5,0,1,"The entity correctly implements the data recording process, but its core weighing behavior is broken; the weighing logic"
3,entity-20-students,⚠️ warn,1,5/0/2,1,2,"The entity successfully models the core behaviors of waste generation, discarding, and event volunteering. However, it r"
4,entity-canonical-0-waste,❌ fail,1,8/4/8,3,2,"The Waste entity is a passive data object that models some state changes correctly via policy methods (e.g., being moved"
5,entity-manual-1777478778843-event,❌ fail,1,0/0/4,0,1,"The entity correctly implements a time-based lifecycle (Scheduled, Active, Completed) and calculates a generic quantity"
6,entity-123-the-waste-cage,⚠️ warn,2,2/2/8,1,2,"The entity correctly models its core function as a waste container with limited capacity, including states for being ful"
7,entity-143-waste-bins,⚠️ warn,2,3/1/5,1,0,"The entity correctly models a passive waste container, implementing core interactions like depositing, overflowing, and"
8,entity-145-waste-collectors,❌ fail,2,2/0/5,3,3,The entity's implementation of its core role—collecting and transporting waste—is detailed and includes a proper state m
9,entity-2-housekeepers,⚠️ warn,2,12/0/1,1,1,"The entity correctly models most core behaviors, including collecting from bins, sorting, inaccurate weighing, and depos"


## 🔬 Pass 1 — Detailed Claim Breakdown

In [ ]:
import pandas as pd

# Normalise pass1_results → list of (eid, score, verdict) regardless of dict or DataFrame
def _iter_pass1(results):
    """Yield (eid, behavior_score, verdict_str) from dict or DataFrame."""
    if isinstance(results, pd.DataFrame):
        for _, row in results.iterrows():
            eid = row.get("entity", "")
            # score column may be int or "?"
            try:
                score = int(row.get("score", 3))
            except (ValueError, TypeError):
                score = 3
            # verdict column has emoji prefix e.g. "✅ pass" → strip to bare word
            raw_verdict = str(row.get("verdict", "pass"))
            verdict = raw_verdict.split()[-1] if raw_verdict else "pass"
            yield eid, score, verdict
    else:
        for eid, r in results.items():
            score = r.get("behavior_score", 3)
            verdict = r.get("verdict", "pass")
            yield eid, score, verdict

print("Pass 1 scores:")
for eid, score, verdict in _iter_pass1(pass1_results):
    print(f"  {eid}: score={score} verdict={verdict}")

print()

flagged_eids = {
    eid for eid, score, verdict in _iter_pass1(pass1_results)
    if score <= PASS2_SCORE_THRESHOLD or verdict != "pass"
}
print(f"Flagged entities (score <= {PASS2_SCORE_THRESHOLD} OR verdict != pass): {flagged_eids}")

flagged_edges = [
    e for e in dep_edges
    if e["from"] in flagged_eids or e["to"] in flagged_eids
]
seen_pairs = set()
unique_flagged_edges = []
for e in flagged_edges:
    pair = (e["from"], e["to"])
    if pair not in seen_pairs:
        seen_pairs.add(pair)
        unique_flagged_edges.append(e)

print(f"Flagged edges for Pass 2: {len(unique_flagged_edges)}")
for e in unique_flagged_edges:
    print(f"  {e['from']} → {e['to']}: {e['reason']}")

In [ ]:
PASS2_SCHEMA = {
    "type": "object",
    "properties": {
        "edge": {"type": "string"},
        "compatible": {"type": "boolean"},
        "issues": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Specific interface mismatches: wrong attr name, missing method, type conflict, etc.",
        },
        "fix_suggestions": {
            "type": "array",
            "items": {"type": "string"},
        },
        "summary": {"type": "string"},
    },
    "required": ["edge", "compatible", "issues", "fix_suggestions", "summary"],
}

def _get_pass1_concerns(results, eid: str) -> str:
    """Extract interface_concerns for an entity from dict or DataFrame."""
    if isinstance(results, pd.DataFrame):
        rows = results[results["entity"] == eid]
        if rows.empty:
            return "None flagged."
        val = rows.iloc[0].get("interface_concerns", "")
        return str(val) if val else "None flagged."
    else:
        r = results.get(eid, {})
        concerns = r.get("interface_concerns", [])
        return "\n".join(concerns) if concerns else "None flagged."

def build_pass2_prompt(edge: dict) -> str:
    from_eid = edge["from"]
    to_eid = edge["to"]
    reason = edge["reason"]

    from_code = entity_index[from_eid]["file"].read_text() if from_eid in entity_index else "(file not found)"
    to_code = entity_index[to_eid]["file"].read_text() if to_eid in entity_index else "(file not found)"

    from_concerns = _get_pass1_concerns(pass1_results, from_eid)
    to_concerns = _get_pass1_concerns(pass1_results, to_eid)

    return f"""\
You are auditing the interface compatibility between two entity classes in a simulation.

## Edge under review
FROM: `{from_eid}` → TO: `{to_eid}`
Causal reason: "{reason}"

## Prior audit concerns
FROM entity concerns: {from_concerns}
TO entity concerns: {to_concerns}

## Your task
Check:
1. Does FROM entity call/interact with TO entity correctly? (method name, action string, payload attrs)
2. Does TO entity's `on_interact()` / `on_query()` accept what FROM entity sends?
3. Are attribute names consistent across both sides?
4. Any type or unit mismatches (e.g., FROM sends kg, TO expects count)?

## FROM entity source: `{from_eid}`

```python
{from_code}
```

## TO entity source: `{to_eid}`

```python
{to_code}
```

Return JSON only.
"""

pass2_results: dict[str, dict] = {}
pass2_errors: dict[str, str] = {}

print(f"Running Pass 2 on {len(unique_flagged_edges)} edges...\n")

for i, edge in enumerate(unique_flagged_edges, 1):
    pair_key = f"{edge['from']} → {edge['to']}"
    print(f"[{i}/{len(unique_flagged_edges)}] {pair_key} ... ", end="", flush=True)

    if edge["from"] not in entity_index or edge["to"] not in entity_index:
        print("SKIP (entity file missing)")
        continue

    try:
        prompt = build_pass2_prompt(edge)
        result = call_gemini(prompt, PASS2_SCHEMA)
        pass2_results[pair_key] = result

        out_path = Path(OUTPUT_DIR) / f"pass2_{edge['from']}__{edge['to']}.json"
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)

        compat = "✅ compatible" if result.get("compatible") else f"❌ {len(result.get('issues', []))} issues"
        print(compat)
    except Exception as exc:
        pass2_errors[pair_key] = str(exc)
        print(f"ERROR: {exc}")

    if i < len(unique_flagged_edges):
        time.sleep(1.5)

print(f"\nDone. {len(pass2_results)} succeeded, {len(pass2_errors)} errors.")

In [ ]:
# Debug: show actual scores from Pass 1
print("Pass 1 scores:")
for eid, r in pass1_results.items():
    score = r.get("behavior_score", "MISSING")
    verdict = r.get("verdict", "MISSING")
    print(f"  {eid}: score={score} verdict={verdict}")

print()

# Flag by EITHER low score OR non-pass verdict (guards against model score/verdict inconsistency)
flagged_eids = {
    eid for eid, r in pass1_results.items()
    if r.get("behavior_score", 3) <= PASS2_SCORE_THRESHOLD
    or r.get("verdict", "pass") != "pass"
}
print(f"Flagged entities (score <= {PASS2_SCORE_THRESHOLD} OR verdict != pass): {flagged_eids}")

flagged_edges = [
    e for e in dep_edges
    if e["from"] in flagged_eids or e["to"] in flagged_eids
]
seen_pairs = set()
unique_flagged_edges = []
for e in flagged_edges:
    pair = (e["from"], e["to"])
    if pair not in seen_pairs:
        seen_pairs.add(pair)
        unique_flagged_edges.append(e)

print(f"Flagged edges for Pass 2: {len(unique_flagged_edges)}")
for e in unique_flagged_edges:
    print(f"  {e['from']} → {e['to']}: {e['reason']}")

In [ ]:
pass1_results

## 🔗 Pass 2 — Cross-Entity Interface Compatibility

In [ ]:
# Identify edges where at least one endpoint is warn/fail
flagged_eids = {
    eid for eid, r in pass1_results.items()
    if r.get("behavior_score", 3) <= PASS2_SCORE_THRESHOLD
}
print(f"Flagged entities (score <= {PASS2_SCORE_THRESHOLD}): {flagged_eids}")

flagged_edges = [
    e for e in dep_edges
    if e["from"] in flagged_eids or e["to"] in flagged_eids
]
# Deduplicate by (from, to) pair
seen_pairs = set()
unique_flagged_edges = []
for e in flagged_edges:
    pair = (e["from"], e["to"])
    if pair not in seen_pairs:
        seen_pairs.add(pair)
        unique_flagged_edges.append(e)

print(f"Flagged edges for Pass 2: {len(unique_flagged_edges)}")
for e in unique_flagged_edges:
    print(f"  {e['from']} → {e['to']}: {e['reason']}")

Flagged entities (score <= 2): set()
Flagged edges for Pass 2: 0


In [ ]:
PASS2_SCHEMA = {
    "type": "object",
    "properties": {
        "edge": {"type": "string"},
        "compatible": {"type": "boolean"},
        "issues": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Specific interface mismatches: wrong attr name, missing method, type conflict, etc.",
        },
        "fix_suggestions": {
            "type": "array",
            "items": {"type": "string"},
        },
        "summary": {"type": "string"},
    },
    "required": ["edge", "compatible", "issues", "fix_suggestions", "summary"],
}

def build_pass2_prompt(edge: dict) -> str:
    from_eid = edge["from"]
    to_eid = edge["to"]
    reason = edge["reason"]

    from_code = entity_index[from_eid]["file"].read_text() if from_eid in entity_index else "(file not found)"
    to_code = entity_index[to_eid]["file"].read_text() if to_eid in entity_index else "(file not found)"

    from_p1 = pass1_results.get(from_eid, {})
    to_p1 = pass1_results.get(to_eid, {})

    from_concerns = "\n".join(from_p1.get("interface_concerns", [])) or "None flagged."
    to_concerns = "\n".join(to_p1.get("interface_concerns", [])) or "None flagged."

    return f"""\
You are auditing the interface compatibility between two entity classes in a simulation.

## Edge under review
FROM: `{from_eid}` → TO: `{to_eid}`
Causal reason: "{reason}"

## Prior audit concerns
FROM entity concerns: {from_concerns}
TO entity concerns: {to_concerns}

## Your task
Check:
1. Does FROM entity call/interact with TO entity correctly? (method name, action string, payload attrs)
2. Does TO entity's `on_interact()` / `on_query()` accept what FROM entity sends?
3. Are attribute names consistent across both sides?
4. Any type or unit mismatches (e.g., FROM sends kg, TO expects count)?

## FROM entity source: `{from_eid}`

```python
{from_code}
```

## TO entity source: `{to_eid}`

```python
{to_code}
```

Return JSON only.
"""

pass2_results: dict[str, dict] = {}
pass2_errors: dict[str, str] = {}

print(f"Running Pass 2 on {len(unique_flagged_edges)} edges...\n")

for i, edge in enumerate(unique_flagged_edges, 1):
    pair_key = f"{edge['from']} → {edge['to']}"
    print(f"[{i}/{len(unique_flagged_edges)}] {pair_key} ... ", end="", flush=True)

    # Skip if either entity file is missing
    if edge["from"] not in entity_index or edge["to"] not in entity_index:
        print("SKIP (entity file missing)")
        continue

    try:
        prompt = build_pass2_prompt(edge)
        result = call_gemini(prompt, PASS2_SCHEMA)
        pass2_results[pair_key] = result

        out_path = Path(OUTPUT_DIR) / f"pass2_{edge['from']}__{edge['to']}.json"
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)

        compat = "✅ compatible" if result.get("compatible") else f"❌ {len(result.get('issues',[]))} issues"
        print(compat)
    except Exception as exc:
        pass2_errors[pair_key] = str(exc)
        print(f"ERROR: {exc}")

    if i < len(unique_flagged_edges):
        time.sleep(1.5)

print(f"\nDone. {len(pass2_results)} succeeded, {len(pass2_errors)} errors.")

## 📊 Pass 2 Results

In [ ]:
rows2 = []
for pair_key, r in pass2_results.items():
    rows2.append({
        "edge": pair_key,
        "compatible": "✅" if r.get("compatible") else "❌",
        "issues": len(r.get("issues", [])),
        "fixes": len(r.get("fix_suggestions", [])),
        "summary": r.get("summary", "")[:140],
    })

if rows2:
    df2 = pd.DataFrame(rows2)
    display(df2)
else:
    print("No Pass 2 results (no flagged edges or all skipped).")

# Detailed issues
for pair_key, r in pass2_results.items():
    if r.get("compatible"):
        continue
    print(f"\n{'='*70}")
    print(f"Edge: {pair_key}")
    print(f"Summary: {r.get('summary','')}")
    for issue in r.get("issues", []):
        print(f"  ❌ {issue}")
    for fix in r.get("fix_suggestions", []):
        print(f"  💡 {fix}")

NameError: name 'pass2_results' is not defined

## 📋 Final Report

## 📐 Evaluation Methodology

### Overview

This notebook evaluates auto-generated simulation entity code using **LLM-as-Judge** — a two-pass Gemini-based pipeline that checks whether each entity's code faithfully implements the causal structure extracted from real-world interviews.

The core problem: code generation produces entities in isolation. Each file may look syntactically correct but fail to implement the causal behaviors the simulation requires, or produce incompatible interfaces when entities interact.

---

### Data Inputs

| Input | Source | Role in evaluation |
|---|---|---|
| Entity source files | `artifacts/entities/*.py` | Code under judgment |
| Dependency graph | `checkpoints/state1c_entity_dependencies.json` | Defines who interacts with whom and why |
| Causal triples | Exported causal bundle JSON (`export_type: causal_bundle`) | Interview-grounded `{head → relationship → tail}` facts |
| Metric contracts | `artifacts/metric_contracts.json` | Declares which attributes each entity must expose for metrics to work |

---

### Pass 1 — Per-Entity Contract Audit

Each entity is judged **independently** against three criteria:

**1. Causal role fulfillment**
The dependency graph provides directed edges with human-readable reasons (e.g. *"housekeepers move waste to cage"*). These are converted into explicit **causal claims** for the judge to verify against the source code. Each claim is rated:

- `FOUND` — behavior clearly present with correct implementation
- `PARTIAL` — method exists but logic is shallow or incomplete
- `MISSING` — no evidence of the behavior in code

**2. Metric attribute coverage**
Metric contracts specify which attributes each entity must expose via `on_query()` (e.g. `current_load`, `sorting_accuracy_ratio`). The judge checks whether each required attribute exists and holds meaningful data.

**3. Interface concerns**
The judge flags any suspicious patterns that could cause cross-entity breakage — wrong action strings, mismatched field names, hardcoded entity IDs — without needing to see the counterpart's code.

**Output per entity:**

| Field | Type | Meaning |
|---|---|---|
| `behavior_score` | 0–3 | Overall implementation completeness |
| `verdict` | pass / warn / fail | Derived from score |
| `causal_claim_checks` | list | Per-claim FOUND / PARTIAL / MISSING + evidence |
| `missing_attrs` | list | Metric attrs absent from `on_query()` |
| `interface_concerns` | list | Potential cross-entity mismatches |

**Scoring rubric:**

| Score | Label | Meaning |
|---|---|---|
| 3 | Complete | All causal roles implemented with correct state transitions |
| 2 | Incomplete | Roles present but logic is shallow or missing edge cases |
| 1 | Partial | Some roles implemented, others absent |
| 0 | Absent | Causal role largely missing from code |

---

### Pass 2 — Cross-Entity Interface Compatibility

Pass 1 cannot detect mismatches that only appear when two entities interact. Pass 2 targets **edges** where at least one endpoint scored ≤ threshold or received a non-pass verdict.

For each flagged edge, both entity files are loaded together and the judge checks:

1. Does the FROM entity call the TO entity with the correct method name and action string?
2. Does the TO entity's `on_interact()` / `on_query()` accept what FROM sends?
3. Are attribute names consistent across both sides?
4. Are there type or unit mismatches (e.g. FROM sends `kg`, TO expects `count`)?

**Output per edge:**

| Field | Type | Meaning |
|---|---|---|
| `compatible` | bool | Whether the interface is sound |
| `issues` | list | Specific mismatches found |
| `fix_suggestions` | list | Concrete code-level fixes |

**Why not check all edges?** With 16 entities the dependency graph has ~20 edges. Checking all pairs unconditionally would use ~40 Gemini calls (2 files each). Filtering to flagged endpoints reduces this to the highest-risk edges while keeping cost manageable. Set `PASS2_SCORE_THRESHOLD = 3` to force a full audit regardless of Pass 1 scores.

---

### Entity Name Resolution

Entity files use ID-based names (e.g. `entity-manual-1777470690931-sorting-facility`). The pipeline resolves human-readable names by:

1. **Priority**: explicit label from `inputs.json` → `userEntityList` (most accurate)
2. **Fallback**: filename parsing — strips known noise prefixes (`entity`, `canonical`, `manual`, `the`) and pure-numeric / UUID segments

This ensures short but meaningful names like `bma` and `n15` are preserved and not silently dropped.

---

### Causal Triple Matching

Each entity receives only the causal triples relevant to it. Matching uses two tiers:

- **Exact substring** — entity name appears inside triple's `head` or `tail` text
- **2+ keyword overlap** — prevents false matches on ubiquitous words like *"waste"* that appear in nearly every triple

Single-keyword match is permitted only when the entity name itself is a single word (e.g. `bma`, `n15`).

---

### Limitations

- **Gemini as judge is not deterministic** — same entity may receive different scores across runs; results should be treated as indicative, not ground truth
- **Metric contract entity names use fuzzy matching** — contracts reference entities by informal names ("Housekeepers", "waste bins") that may not match exactly; mismatches cause missed or spurious attr checks
- **Pass 1 is context-blind** — an entity that delegates behavior to policies (rather than implementing it in `step()`) may be rated lower than deserved, since the policy code is not included in the audit
- **Pass 2 only covers flagged edges** — interface bugs in entities that both scored 3 will not be caught unless threshold is raised

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# ── Normalise pass1 input ────────────────────────────────────────────────────
def _to_eval_df(results) -> pd.DataFrame:
    """Convert pass1_results (dict or DataFrame) to a clean eval DataFrame."""
    if isinstance(results, pd.DataFrame):
        df = results.copy()
        # Normalise verdict: strip emoji prefix → bare word
        df["verdict_clean"] = df["verdict"].astype(str).str.split().str[-1]
        # Score as int
        df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(3).astype(int)
        return df
    else:
        rows = []
        for eid, r in results.items():
            claims = r.get("causal_claim_checks", [])
            found   = sum(1 for c in claims if c["status"] == "FOUND")
            partial = sum(1 for c in claims if c["status"] == "PARTIAL")
            missing = sum(1 for c in claims if c["status"] == "MISSING")
            rows.append({
                "entity": eid,
                "score": r.get("behavior_score", 3),
                "verdict": r.get("verdict", "pass"),
                "verdict_clean": r.get("verdict", "pass"),
                "claims (F/P/M)": f"{found}/{partial}/{missing}",
                "missing_attrs": len(r.get("missing_attrs", [])),
                "interface_concerns": len(r.get("interface_concerns", [])),
                "summary": r.get("summary", ""),
            })
        return pd.DataFrame(rows)

df_eval = _to_eval_df(pass1_results)

# ── METHOD OVERVIEW ──────────────────────────────────────────────────────────
display(Markdown("""
### Method

| Pass | What is judged | Input per request | Output |
|------|---------------|-------------------|--------|
| **Pass 1** | Each entity in isolation against its causal role + metric contracts | Entity source code + dependency edges + causal triples + required metric attrs | `behavior_score` 0–3, `verdict` pass/warn/fail, per-claim FOUND/PARTIAL/MISSING |
| **Pass 2** | Cross-entity interface compatibility for flagged pairs | Both entity source files + Pass 1 concerns | `compatible` bool, `issues[]`, `fix_suggestions[]` |

**Scoring rubric (behavior_score)**

| Score | Meaning |
|-------|---------|
| 3 | All causal roles implemented, correct state transitions |
| 2 | Roles present but incomplete or shallow |
| 1 | Partially missing — some roles absent |
| 0 | Causal role largely absent from code |

**Flagging rule for Pass 2:** score ≤ threshold **OR** verdict ≠ pass
"""))

# ── AGGREGATE STATS ──────────────────────────────────────────────────────────
total   = len(df_eval)
n_pass  = (df_eval["verdict_clean"] == "pass").sum()
n_warn  = (df_eval["verdict_clean"] == "warn").sum()
n_fail  = (df_eval["verdict_clean"] == "fail").sum()
avg_score = df_eval["score"].mean()

display(Markdown(f"""
### Aggregate Results — Pass 1 ({total} entities)

| | Count | % |
|---|---|---|
| ✅ Pass | {n_pass} | {n_pass/total*100:.0f}% |
| ⚠️ Warn | {n_warn} | {n_warn/total*100:.0f}% |
| ❌ Fail | {n_fail} | {n_fail/total*100:.0f}% |
| **Average score** | **{avg_score:.2f} / 3** | |
"""))

# ── PER-ENTITY TABLE ─────────────────────────────────────────────────────────
display(Markdown("### Per-Entity Verdicts"))

show_cols = ["entity", "score", "verdict", "claims (F/P/M)", "missing_attrs", "interface_concerns", "summary"]
available = [c for c in show_cols if c in df_eval.columns]

styled = (
    df_eval[available]
    .sort_values("score")
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", 100)
display(styled)

# ── SCORE DISTRIBUTION ───────────────────────────────────────────────────────
display(Markdown("### Score Distribution"))
dist = df_eval.groupby("score")["entity"].count().reset_index()
dist.columns = ["score", "entity_count"]
dist["verdict_label"] = dist["score"].map({0: "absent", 1: "partial", 2: "incomplete", 3: "complete"})
display(dist)

# ── TOP ISSUES PATTERN ───────────────────────────────────────────────────────
# Parse claims column if it exists as "F/P/M" string
if "claims (F/P/M)" in df_eval.columns:
    def parse_fpm(s):
        try:
            f, p, m = str(s).split("/")
            return int(f), int(p), int(m)
        except Exception:
            return 0, 0, 0

    df_eval[["_found","_partial","_missing"]] = df_eval["claims (F/P/M)"].apply(
        lambda s: pd.Series(parse_fpm(s))
    )
    total_found   = df_eval["_found"].sum()
    total_partial = df_eval["_partial"].sum()
    total_missing = df_eval["_missing"].sum()
    total_claims  = total_found + total_partial + total_missing

    display(Markdown(f"""
### Causal Claim Coverage (across all entities)

| Status | Count | % of all claims |
|--------|-------|-----------------|
| FOUND   | {total_found}  | {total_found/total_claims*100:.1f}% |
| PARTIAL | {total_partial} | {total_partial/total_claims*100:.1f}% |
| MISSING | {total_missing} | {total_missing/total_claims*100:.1f}% |
| **Total claims** | **{total_claims}** | |
"""))

# ── PASS 2 SUMMARY (if available) ────────────────────────────────────────────
if "pass2_results" in dir() and pass2_results:
    n_edges   = len(pass2_results)
    n_incompat = sum(1 for r in pass2_results.values() if not r.get("compatible", True))
    display(Markdown(f"""
### Pass 2 — Interface Compatibility ({n_edges} edges audited)

| | Count |
|---|---|
| ✅ Compatible | {n_edges - n_incompat} |
| ❌ Incompatible | {n_incompat} |
"""))
else:
    display(Markdown("### Pass 2 — Not yet run or no flagged edges"))

## 📐 Evaluation Methodology & Findings